### Read the ERP customer data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Read Bronze table


In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")


# Display the raw Bronze data first

In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. The customer key contains two formats, and some values start with NAS, so it needs to be standardized.
 3. The birth date column needs to be converted to a proper date type.
 4. Some birth date values are invalid or future dates and should be cleaned.
 5. The gender column contains null, blank, and short code values such as M and F.
 6. We should check for duplicate customer records based on the business key.
 7. Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.

============================================================

# Silver Transformations


# Trimming

In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Validation checks before cleaning



In [0]:
# Validation checks before cleaning ERP customer data
df.select(
    F.count(F.when(col("CID").isNull(), True)).alias("null_customer_key_count"),
    F.count(F.when(col("CID").startswith("NAS"), True)).alias("customer_key_with_nas_prefix_count"),
    F.count(F.when(F.to_date(col("BDATE"), "yyyy-MM-dd").isNull(), True)).alias("invalid_birth_date_count"),
    F.count(F.when(F.to_date(col("BDATE"), "yyyy-MM-dd") > F.current_date(), True)).alias("future_birth_date_count"),
    F.count(F.when(col("GEN").isNull() | (trim(col("GEN")) == ""), True)).alias("null_or_blank_gender_count")
).display()

# Customer Key Parsing


In [0]:
# Standardizing the customer key to match the CRM customer key format
df = df.withColumn("CID", F.regexp_replace(col("CID"), "^NAS", ""))



# Cleaning Birth Date

In [0]:
# Converting birth date to proper date format
df = df.withColumn("BDATE", F.to_date(col("BDATE"), "yyyy-MM-dd"))

# Replacing future birth dates with null
df = df.withColumn(
    "BDATE",
    F.when(col("BDATE") > F.current_date(), None)
     .otherwise(col("BDATE"))
)

# Standardizing Gender


In [0]:
# Standardizing gender values
df = df.withColumn(
    "GEN",
    F.when(F.upper(trim(col("GEN"))) == "M", "Male")
     .when(F.upper(trim(col("GEN"))) == "F", "Female")
     .when(F.upper(trim(col("GEN"))) == "MALE", "Male")
     .when(F.upper(trim(col("GEN"))) == "FEMALE", "Female")
     .otherwise("n/a")
)

# Validation checks after cleaning



In [0]:
# Validation checks after cleaning ERP customer data
df.select(
    F.count(F.when(col("CID").isNull(), True)).alias("null_customer_key_count"),
    F.count(F.when(~col("CID").rlike("^AW[0-9]{8}$"), True)).alias("invalid_customer_key_count"),
    F.count(F.when(col("BDATE").isNull(), True)).alias("null_birth_date_count"),
    F.count(F.when(col("BDATE") > F.current_date(), True)).alias("future_birth_date_count"),
    F.count(F.when(col("GEN") == "n/a", True)).alias("unknown_gender_count")
).display()

# check duplicate record on bussiness keys in this file



In [0]:
# Check for duplicate records based on the business key
# to make sure the customer rows are unique before saving to Silver.
df.groupBy("CID") \
  .count() \
  .filter(col("count") > 1) \
  .display()

# Removing duplicates


In [0]:
# Removing duplicate customer records based on the business key
df = df.dropDuplicates(["CID"])

# Validation after removing duplicates


In [0]:
# Validation checks after removing duplicate customer records
df.groupBy("CID") \
  .count() \
  .filter(col("count") > 1) \
  .count()

#rename coulmns



In [0]:
# Renaming columns to cleaner business names
RENAME_MAP = {
    "CID": "customer_key",
    "BDATE": "birth_date",
    "GEN": "gender"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#quick validation for df


In [0]:
# Display the cleaned dataframe for a quick validation check
df.display()

#writing silver table



In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

#check table created or not



In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.erp_customers LIMIT 10